# CH04 -- Notebook thực hành
Chạy lần lượt từng ô từ trên xuống. Đặt notebook ở thư mục gốc dự án (cùng cấp với thư mục `data/` và `figures/`).

### Ô lệnh 1 -- Nạp thư viện, dữ liệu Credit Default và bộ dữ liệu "đồ chơi"

In [ ]:
import sys
sys.path.append("code")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from common import credit_split, xgb_clf

pd.set_option("display.width", 80)
pd.set_option("display.precision", 3)
SEED = 42

X_train, X_test, y_train, y_test = credit_split()
X_toy, y_toy = make_classification(n_samples=600, n_features=2, n_redundant=0,
                                   n_clusters_per_class=1, weights=[0.9, 0.1],
                                   class_sep=1.0, random_state=3)
print("Credit -- train:", X_train.shape, "| Đồ chơi:", X_toy.shape, np.bincount(y_toy))


### Ô lệnh 2 -- Đo "độ khó" r_i của một điểm thiểu số bằng 5 láng giềng

In [ ]:
K = 5
nn_all = NearestNeighbors(n_neighbors=K + 1).fit(X_toy)     # tìm trong TOÀN BỘ dữ liệu
min_idx = np.where(y_toy == 1)[0]

for i in min_idx[[1, 32, 21]]:                              # ba điểm đã gặp ở Chương 3
    nb = nn_all.kneighbors(X_toy[[i]], return_distance=False)[0][1:]
    delta = (y_toy[nb] == 0).sum()                          # số láng giềng thuộc lớp đa số
    print(f"Điểm {i:3d}: nhãn láng giềng = {y_toy[nb]} -> Delta = {delta}, r = {delta / K:.1f}")


### Ô lệnh 3 -- Chia "ngân sách" G điểm tổng hợp cho từng điểm thiểu số

In [ ]:
def adasyn_weights(X, y, K=5):
    """Tính r_i (độ khó), r_hat_i (đã chuẩn hóa) và g_i (số điểm sẽ sinh từ điểm i)."""
    G = (y == 0).sum() - (y == 1).sum()                     # tổng số điểm cần sinh
    nb = NearestNeighbors(n_neighbors=K + 1).fit(X).kneighbors(
        X[y == 1], return_distance=False)[:, 1:]
    r = (y[nb] == 0).sum(axis=1) / K
    r_hat = r / r.sum()                                     # tổng các r_hat bằng 1
    g = np.rint(r_hat * G).astype(int)                      # làm tròn thành số nguyên
    return r, r_hat, g, G

r, r_hat, g, G = adasyn_weights(X_toy, y_toy)
print(f"G = {G}, tổng các r_i = {r.sum():.1f}, tổng các g_i = {g.sum()}")
print(pd.DataFrame({"r_i": r, "r_hat_i": r_hat, "g_i": g}).iloc[[1, 32, 21]])
print("Số điểm có g_i = 0:", (g == 0).sum(), "/", len(g), "| g_i lớn nhất:", g.max())


### Ô lệnh 4 -- Vẽ: màu = độ khó r_i, kích thước = số điểm g_i được sinh

In [ ]:
X_min = X_toy[y_toy == 1]
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.scatter(*X_toy[y_toy == 0].T, s=8, color="#c9c8c2")
sc = ax.scatter(*X_min.T, c=r, cmap="Oranges", vmin=0, vmax=1, s=10 + g * 1.5,
                edgecolor="#52514e", linewidth=0.4)
fig.colorbar(sc, ax=ax, label="r_i (tỷ lệ láng giềng đa số)", shrink=0.85)
plt.savefig("figures/ch04_s04_weights.pdf", bbox_inches="tight")


### Ô lệnh 5 -- Tự viết ADASYN: mỗi điểm sinh đúng g_i điểm mới

In [ ]:
def adasyn_tay(X_min, g, k=5, seed=SEED):
    rng = np.random.default_rng(seed)
    _, idx = NearestNeighbors(n_neighbors=k + 1).fit(X_min).kneighbors(X_min)
    new = []
    for i in range(len(X_min)):                  # KHÁC SMOTE: duyệt từng điểm,
        for _ in range(g[i]):                    # mỗi điểm sinh đúng g_i lần
            j = rng.choice(idx[i][1:])
            lam = rng.uniform(0, 1)
            new.append(X_min[i] + lam * (X_min[j] - X_min[i]))
    return np.array(new)

X_syn = adasyn_tay(X_min, g)
print("Số điểm sinh ra:", len(X_syn))
top = np.argsort(-g)[:3]
print("Ba điểm sinh nhiều nhất:", {int(i): int(g[i]) for i in top},
      "-> chiếm", f"{g[top].sum() / g.sum():.0%}", "tổng số điểm mới")


### Ô lệnh 6 -- So sánh SMOTE, Borderline-1 và ADASYN (thư viện imblearn)

In [ ]:
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN

samplers = {"SMOTE": SMOTE(random_state=SEED),
            "Borderline-1": BorderlineSMOTE(kind="borderline-1", random_state=SEED),
            "ADASYN": ADASYN(n_neighbors=5, random_state=SEED)}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharex=True, sharey=True)
for ax, (name, s) in zip(axes, samplers.items()):
    Xr, yr = s.fit_resample(X_toy, y_toy)
    ax.scatter(*X_toy[y_toy == 0].T, s=6, color="#c9c8c2")
    ax.scatter(*X_min.T, s=12, color="#eb6834")
    ax.scatter(*Xr[len(X_toy):].T, s=8, marker="x", color="#2a78d6", linewidth=0.7)
    ax.set_title(f"{name} ({len(Xr) - len(X_toy)} điểm mới)")
plt.savefig("figures/ch04_s06_compare.pdf", bbox_inches="tight")


### Ô lệnh 7 -- Ngân sách đi về đâu? So sánh ba cách phân bổ (cùng dùng 5 láng giềng)

In [ ]:
groups = pd.cut(r, bins=[-0.01, 0.49, 0.99, 1.0], labels=["r < 0,5", "0,5 <= r < 1", "r = 1"])
share = pd.DataFrame({
    "Số điểm": pd.Series(groups).value_counts().sort_index(),
    "SMOTE": pd.Series(np.ones(len(r)), index=groups).groupby(level=0, observed=False).sum() / len(r),
    "Kiểu Borderline": pd.Series(((r >= 0.5) & (r < 1)).astype(float), index=groups)
                    .groupby(level=0, observed=False).sum() / ((r >= 0.5) & (r < 1)).sum(),
    "ADASYN": pd.Series(g, index=groups).groupby(level=0, observed=False).sum() / g.sum(),
})
print(share.round(2))


### Ô lệnh 8 -- Trên Credit Default: phân bố r_i và ngân sách ADASYN đi về đâu

In [ ]:
Z_train = StandardScaler().fit_transform(X_train)
rc, _, gc, Gc = adasyn_weights(Z_train, y_train.values)
tab = pd.DataFrame({"% khách hàng vỡ nợ": pd.Series(rc).value_counts(normalize=True).sort_index() * 100,
                    "% điểm tổng hợp": pd.Series(gc).groupby(rc).sum() / gc.sum() * 100})
tab.index.name = "r_i"
print(tab.round(1))

fig, ax = plt.subplots(1, 2, figsize=(8, 2.8), sharey=True)
ax[0].bar(tab.index, tab.iloc[:, 0], width=0.15, color="#2a78d6")
ax[0].set_title("Khách hàng vỡ nợ theo r_i"); ax[0].set_ylabel("%")
ax[1].bar(tab.index, tab.iloc[:, 1], width=0.15, color="#eb6834")
ax[1].set_title("Điểm tổng hợp sinh ra từ đâu")
for a in ax:
    a.set_xlabel("r_i")
plt.savefig("figures/ch04_s08_credit_r.pdf", bbox_inches="tight")


### Ô lệnh 9 -- Đánh giá chéo: không lấy mẫu, SMOTE, ADASYN (2 mô hình)

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {"PR-AUC": "average_precision", "F1": "f1", "Recall": "recall", "Precision": "precision"}

def make(model, sampler):
    steps = [("scale", StandardScaler())]
    if sampler is not None:
        steps.append(("sampler", sampler))
    return Pipeline(steps + [("model", model)])

strategies = {"Không lấy mẫu": None, "SMOTE": samplers["SMOTE"], "ADASYN": samplers["ADASYN"]}
cv_rows = {}
for mname, mk in {"LogReg": lambda: LogisticRegression(max_iter=2000), "XGBoost": xgb_clf}.items():
    for sname, s in strategies.items():
        r_ = cross_validate(make(mk(), s), X_train, y_train, cv=skf, scoring=scoring)
        cv_rows[f"{mname} | {sname}"] = {k: r_["test_" + k].mean() for k in scoring}
print(pd.DataFrame(cv_rows).T)


### Ô lệnh 10 -- Ảnh hưởng của số láng giềng K (XGBoost + ADASYN)

In [ ]:
rows = []
for K_ in [3, 5, 10, 20]:
    r_ = cross_validate(make(xgb_clf(), ADASYN(n_neighbors=K_, random_state=SEED)),
                        X_train, y_train, cv=skf, scoring=scoring)
    rows.append({"K": K_, **{k: r_["test_" + k].mean() for k in scoring}})
print(pd.DataFrame(rows).set_index("K"))


### Ô lệnh 11 -- Thử thách: dữ liệu train có NHÃN SAI. Kỹ thuật nào chịu đựng tốt nhất?

In [ ]:
from sklearn.metrics import average_precision_score, f1_score

rng = np.random.default_rng(SEED)
y_noisy = y_train.copy()
flip = rng.choice(np.where(y_train == 0)[0], size=int(0.05 * (y_train == 0).sum()), replace=False)
y_noisy.iloc[flip] = 1                     # 5% người KHÔNG vỡ nợ bị ghi nhầm thành vỡ nợ
print(f"Đã lật nhãn {len(flip)} khách hàng (chỉ trong tập train; tập test giữ nguyên)")

rows = {}
for sname, s in {"Không lấy mẫu": None, **samplers}.items():
    res = {}
    for tag, yy in [("sạch", y_train), ("nhiễu", y_noisy)]:
        p = make(xgb_clf(), s).fit(X_train, yy).predict_proba(X_test)[:, 1]
        res[f"PR-AUC {tag}"] = average_precision_score(y_test, p)
        res[f"F1 {tag}"] = f1_score(y_test, (p >= 0.5).astype(int))
    res["Giảm F1"] = res["F1 sạch"] - res["F1 nhiễu"]
    rows[sname] = res
print(pd.DataFrame(rows).T.round(3).to_string())


### Ô lệnh 12 -- Bảng tổng kết Chương 2-4 trên tập test (XGBoost)

In [ ]:
from sklearn.metrics import recall_score, precision_score

final = {}
for sname, s in {"Không lấy mẫu": None, **samplers,
                 "Borderline-2": BorderlineSMOTE(kind="borderline-2", random_state=SEED)}.items():
    p = make(xgb_clf(), s).fit(X_train, y_train).predict_proba(X_test)[:, 1]
    final[sname] = (p, 0.5)
    if s is None:
        final["Không lấy mẫu, t = 0,26"] = (p, 0.26)
summary = {}
for name, (p, t) in final.items():
    pred = (p >= t).astype(int)
    summary[name] = {"F1": f1_score(y_test, pred), "Recall": recall_score(y_test, pred),
                     "Precision": precision_score(y_test, pred),
                     "PR-AUC": average_precision_score(y_test, p)}
print(pd.DataFrame(summary).T.round(3))
